# 📚 MIDA507 -- Session 01
## Explorer le Big Data avec des Données Documentaires Africaines

| | |
|---|---|
| **Programme** | Master MIDA -- Université de Douala |
| **Session** | S01 -- Fondements du Big Data et de la Data Science |
| **Durée estimée** | 90 à 120 minutes |
| **Niveau Python requis** | Aucun -- tout est expliqué pas à pas |

### Ce que vous allez apprendre
1. Ce qu'est le Big Data et pourquoi ce concept concerne l'IDA
2. Les 7 caractéristiques (7V) du Big Data mesurées sur un vrai jeu africain
3. Comment Python peut aider un professionnel de l'information sans qu'il soit programmeur
4. Le rôle central de l'IDA dans le pipeline de données numériques

### Livrable de la session
`MIDA507_S01_rapport_7V.json` -- Votre analyse des 7V appliquée à un jeu africain


## 👋 Bienvenue dans votre premier Notebook Google Colab

Avant de commencer, quelques repères essentiels :

| Type de cellule | Apparence | Ce que vous faites |
|---|---|---|
| **Cellule de texte** (comme celle-ci) | Fond blanc | Lisez-la -- elle explique une notion |
| **Cellule de code** | Fond gris | Exécutez-la en appuyant sur ▶️ ou **Shift + Entrée** |

### Règle d'or
> Exécutez toujours les cellules **dans l'ordre, de haut en bas**.
> Vous n'avez pas besoin de comprendre le code -- ce notebook l'explique pour vous.
> Votre rôle : **lire les explications, observer les résultats, répondre aux exercices**.

---


In [ ]:
# ================================================================
# INSTALLATION DES OUTILS -- A exécuter UNE SEULE FOIS par session
# ================================================================
# Python utilise des "bibliothèques" : des boîtes à outils pré-programmées.
# La commande !pip install télécharge et installe ces boîtes.
# Des messages vont défiler -- c'est normal. Attendez "Installation terminée".
# ================================================================

!pip install pandas --quiet        # Outil pour manipuler des tableaux de données
!pip install seaborn --quiet       # Outil pour créer des graphiques statistiques
!pip install matplotlib --quiet    # Outil graphique de base (courbes, histogrammes)
!pip install openpyxl --quiet      # Outil pour lire et écrire des fichiers Excel

print()
print("=" * 55)
print("  ✅ INSTALLATION TERMINÉE")
print("  Tous les outils sont prêts à l'emploi.")
print("  Passez à la cellule suivante (Shift + Entrée).")
print("=" * 55)


In [ ]:
# ================================================================
# CHARGEMENT DES OUTILS EN MÉMOIRE
# ================================================================
# "import" signifie : "charge cet outil pour que je puisse l'utiliser".
# C'est comme ouvrir vos boîtes à outils avant de commencer un travail.
# Le mot après "as" est un surnom court pour aller plus vite à écrire.
# ================================================================

import pandas as pd            # "pd"  = surnom de pandas
import seaborn as sns           # "sns" = surnom de seaborn
import matplotlib.pyplot as plt # "plt" = surnom de matplotlib
import json                     # Pour lire/écrire des fichiers JSON
import os                       # Pour gérer les fichiers sur le disque
import hashlib                  # Pour calculer des empreintes numériques (checksums)
import random                   # Pour générer des valeurs aléatoires
from datetime import datetime, timedelta   # Pour manipuler les dates

# Réglages de l'affichage des graphiques
plt.rcParams['figure.figsize'] = (12, 5)   # Taille par défaut des graphiques
sns.set_theme(style="whitegrid")            # Style visuel : fond blanc avec quadrillage

print("✅ Tous les outils sont chargés en mémoire.")
print("   Nous sommes prêts à travailler.")


---
## 📖 NOTION 1 -- Qu'est-ce que le Big Data ?

### Définition en 3 lignes

Le **Big Data** désigne des ensembles de données si volumineux, variés ou produits si rapidement qu'ils dépassent la capacité des outils classiques (Excel, Word) à les stocker, gérer et analyser.

**Analogie IDA connue :** C'est comme passer d'un fonds documentaire de 500 fiches cartonnées (gérable manuellement) à un fonds de 5 millions de notices numériques produites chaque jour par des capteurs, des formulaires, des réseaux sociaux et des systèmes de gestion.

### Pourquoi ça concerne l'IDA ?

Les bibliothèques, centres de documentation et archives africains génèrent de plus en plus de données numériques : fichiers d'emprunts, statistiques d'usage, métadonnées de numérisation, journaux d'accès en ligne. Le professionnel IDA doit savoir **décrire, évaluer et gérer** ces volumes, même sans être informaticien.

### Le concept des 7V

Les chercheurs ont identifié **7 caractéristiques** (appelées "7V") pour décrire et évaluer un jeu de Big Data. Nous allons mesurer chacun de ces V sur notre jeu de la BU-UNG.

| V | Nom | Question clé |
|---|---|---|
| V1 | **Volume** | Quelle est la taille du jeu ? |
| V2 | **Vélocité** | À quelle vitesse les données sont-elles produites ? |
| V3 | **Variété** | Quels types de données coexistent ? |
| V4 | **Véracité** | Les données sont-elles fiables et complètes ? |
| V5 | **Valeur** | Quelles décisions ces données permettent-elles ? |
| V6 | **Variabilité** | Le sens des données change-t-il selon le contexte ? |
| V7 | **Visualisation** | Comment rendre les données lisibles pour les décideurs ? |


In [ ]:
# ================================================================
# CRÉATION DU JEU DE DONNÉES FIL CONDUCTEUR
# ================================================================
# Tout au long des sessions MIDA507, nous travaillons sur LE MÊME
# jeu de données : les emprunts de la Bibliothèque Centrale de
# l'Université de Ngaoundéré (BU-UNG) entre 2018 et 2023.
#
# Ce jeu est FICTIF mais basé sur des réalités camerounaises authentiques.
# Il contient 5 000 enregistrements et 10 colonnes.
#
# En situation réelle dans votre institution, vous remplaceriez
# tout ce bloc par une seule ligne :
#   emprunts = pd.read_csv("votre_fichier.csv")
# ================================================================

# Graine aléatoire fixe : garantit que les données sont identiques
# à chaque exécution (reproductibilité -- notion vue en S02)
random.seed(42)

NB = 5000   # Nombre d'enregistrements

# -- Valeurs possibles pour chaque colonne --
TYPES_DOCS  = ["Thèse et mémoire","Manuel universitaire","Revue scientifique",
               "Rapport de recherche","Ouvrage de référence","Document administratif"]
POIDS_TYPES = [0.28, 0.30, 0.15, 0.10, 0.10, 0.07]

FILIERES    = ["Droit","Sciences économiques","Lettres modernes","Histoire",
               "Géographie","Médecine","Agronomie","Informatique"]

NIVEAUX     = ["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]

REGIONS     = ["Adamaoua","Centre","Est","Extrême-Nord",
               "Littoral","Nord","Ouest","Sud"]

LANGUES     = ["Français","Anglais","Bilingue FR/EN","Arabe","Autres langues africaines"]
POIDS_LANG  = [0.62, 0.22, 0.10, 0.04, 0.02]

# Génération des dates d'emprunt (réparties sur 5 ans)
d0 = datetime(2018, 1, 1)
dates_emprunt = [
    (d0 + timedelta(days=random.randint(0, 365*5))).strftime("%Y-%m-%d")
    for _ in range(NB)
]

# Construction du tableau (DataFrame) avec toutes les colonnes
emprunts = pd.DataFrame({
    "cote_document":  [f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_document":  random.choices(TYPES_DOCS,  weights=POIDS_TYPES, k=NB),
    "date_emprunt":   dates_emprunt,
    "duree_jours":    random.choices([3,7,7,14,14,14,21,30], k=NB),
    "code_usager":    [f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":        random.choices(FILIERES, k=NB),
    "niveau_etude":   random.choices(NIVEAUX,  k=NB),
    "region_origine": random.choices(REGIONS,  k=NB),
    "langue_document":random.choices(LANGUES,  weights=POIDS_LANG, k=NB),
})

# Ajout de l'année pour faciliter les analyses par année
emprunts["annee"] = pd.to_datetime(emprunts["date_emprunt"]).dt.year

# -- Affichage du résultat --
print("✅ Jeu de données BU-UNG créé avec succès !")
print()
print(f"   Nombre de lignes    : {len(emprunts):,} enregistrements d'emprunt")
print(f"   Nombre de colonnes  : {len(emprunts.columns)} informations par emprunt")
print(f"   Période couverte    : {emprunts['date_emprunt'].min()} au {emprunts['date_emprunt'].max()}")
print(f"   Usagers distincts   : {emprunts['code_usager'].nunique():,} usagers identifiés")
print()
print("   Colonnes du tableau :")
for i, col in enumerate(emprunts.columns, 1):
    print(f"     {i:2}. {col}")


In [ ]:
# ================================================================
# APERÇU DU TABLEAU DE DONNÉES
# ================================================================
# La commande .head(5) affiche les 5 premières lignes du tableau.
# C'est TOUJOURS la première chose à faire quand on découvre un jeu.
# C'est l'équivalent de "feuilleter rapidement un fichier cartonné".
# ================================================================

print("LES 5 PREMIÈRES LIGNES DU JEU DE DONNÉES BU-UNG :")
print("-" * 55)

# .head(5) = les 5 premières lignes
# Dans Colab, un DataFrame s'affiche automatiquement sous forme de tableau
emprunts.head(5)


---
## 💻 V1 -- VOLUME : Quelle est la taille du jeu de données ?

### La notion en une phrase
Le **Volume** mesure la quantité de données : nombre d'enregistrements, taille en mémoire, nombre de variables.

**Analogie IDA :** C'est comme compter le nombre de boîtes d'archives dans un dépôt, le nombre de fiches dans un fichier papier, ou le nombre de notices dans un catalogue informatisé.

### Pourquoi mesurer le volume ?
Le volume détermine les outils nécessaires. 500 enregistrements = gérable sous Excel. 5 000 000 = nécessite Python, des bases de données ou des outils Big Data dédiés.

### En pratique -- mesurons le volume de notre jeu BU-UNG 👇


In [ ]:
# ================================================================
# MESURE DU VOLUME -- V1 du Big Data
# ================================================================
# len() = fonction Python qui compte le nombre d'éléments
# .columns = la liste des colonnes d'un tableau
# .memory_usage() = la taille en mémoire de chaque colonne
# ================================================================

# -- Calcul du nombre de lignes (= nombre d'enregistrements) --
nombre_emprunts = len(emprunts)   # len() compte les lignes

# -- Calcul du nombre de colonnes (= nombre d'informations par emprunt) --
nombre_colonnes = len(emprunts.columns)

# -- Calcul de la taille totale en mémoire --
# .sum() additionne les tailles de toutes les colonnes
# / 1024 / 1024 convertit les octets en mégaoctets (Mo)
taille_en_mo = emprunts.memory_usage(deep=True).sum() / 1024 / 1024

# -- Calcul du nombre d'usagers distincts --
nb_usagers_distincts = emprunts["code_usager"].nunique()

# -- Affichage des résultats --
print("V1 -- VOLUME du jeu de données BU-UNG")
print("=" * 45)
print(f"  Nombre d'enregistrements   : {nombre_emprunts:,} emprunts")
print(f"  Nombre de colonnes         : {nombre_colonnes} informations par emprunt")
print(f"  Taille en mémoire          : {taille_en_mo:.2f} Mo")
print(f"  Usagers distincts          : {nb_usagers_distincts:,} usagers")
print()
print("  INTERPRÉTATION IDA :")
print(f"  → Sur 5 ans, la BU-UNG a enregistré {nombre_emprunts:,} transactions.")
print(f"  → Ce volume dépasse la capacité d'Excel (limite : ~1 million de lignes).")
print(f"  → Un tableur classique aurait des difficultés à analyser cette quantité.")
print(f"  → Python traite ces {nombre_emprunts:,} lignes en moins d'une seconde.")


---
## 💻 V2 -- VÉLOCITÉ : À quelle vitesse les données sont-elles produites ?

### La notion en une phrase
La **Vélocité** mesure la fréquence de production et d'accumulation des données dans le temps.

**Analogie IDA :** Dans un fichier papier, vous comptez les nouvelles fiches ajoutées par jour. En numérique, un SIGB (Système Intégré de Gestion de Bibliothèque) peut enregistrer des centaines d'emprunts par heure en période d'examens.

### En pratique -- mesurons la vélocité de notre jeu 👇


In [ ]:
# ================================================================
# MESURE DE LA VÉLOCITÉ -- V2 du Big Data
# ================================================================
# .groupby("annee") = regroupe les emprunts par année
# .size() = compte le nombre d'emprunts dans chaque groupe
# ================================================================

# -- Calcul du nombre d'emprunts par année --
# On regroupe le tableau par la colonne "annee" et on compte
emprunts_par_annee = emprunts.groupby("annee").size()

# -- Calcul de la vitesse moyenne par jour --
# Le jeu couvre 5 ans = 365 jours x 5 = 1825 jours
jours_total = 365 * 5
emprunts_par_jour = len(emprunts) / jours_total

# -- Affichage des résultats --
print("V2 -- VÉLOCITÉ : rythme de production des données")
print("=" * 50)
print()
print("  Nombre d'emprunts par année :")
for annee, nb in emprunts_par_annee.items():
    barre = "█" * (nb // 80)   # Barre visuelle proportionnelle
    print(f"    {annee} : {nb:,} emprunts  {barre}")
print()
print(f"  Vitesse moyenne : {emprunts_par_jour:.1f} emprunts enregistrés par jour")
print()
print("  INTERPRÉTATION IDA :")
print(f"  → La BU-UNG produit en moyenne {emprunts_par_jour:.0f} nouvelles fiches d'emprunt par jour.")
print(f"  → En période d'examens, ce chiffre peut doubler ou tripler.")
print(f"  → Défi IDA : archiver et indexer ces données en temps réel.")
print(f"  → Recommandation : export mensuel automatique du SIGB + archivage CSV.")

# -- Graphique de l'évolution annuelle --
fig, ax = plt.subplots()
emprunts_par_annee.plot(kind='bar', ax=ax, color='#BF360C', edgecolor='white')
ax.set_title("V2 -- Vélocité : Nombre d'emprunts par année (BU-UNG 2018-2023)",
             fontsize=12, fontweight='bold', color='#3E1500')
ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'emprunts")
ax.tick_params(axis='x', rotation=0)

# Affichage des valeurs au-dessus de chaque barre
for i, (annee, val) in enumerate(emprunts_par_annee.items()):
    ax.text(i, val + 10, str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.show()


---
## 💻 V3 -- VARIÉTÉ : Quels types de données coexistent ?

### La notion en une phrase
La **Variété** décrit la diversité des types de données dans un même jeu : formats différents, catégories multiples, langues variées.

**Analogie IDA :** Dans une bibliothèque, le fonds contient des livres, des thèses, des revues, des cartes, des documents numériques -- chacun avec son type de description (catalogage). La variété numérique reflète cette même hétérogénéité.

### En pratique -- mesurons la variété de notre jeu 👇


In [ ]:
# ================================================================
# MESURE DE LA VARIÉTÉ -- V3 du Big Data
# ================================================================
# .value_counts() = compte combien de fois chaque valeur apparaît
# .unique() = liste les valeurs distinctes sans répétition
# .nunique() = compte combien de valeurs distinctes il y a
# ================================================================

# -- Variété des types de documents --
variete_types = emprunts["type_document"].value_counts()

# -- Variété des langues --
variete_langues = emprunts["langue_document"].value_counts()

# -- Variété des filières --
nb_filieres = emprunts["filiere"].nunique()

# -- Variété des régions --
nb_regions = emprunts["region_origine"].nunique()

print("V3 -- VARIÉTÉ : diversité du jeu de données")
print("=" * 50)
print()
print("  Types de documents présents :")
for type_doc, nb in variete_types.items():
    pourcentage = nb / len(emprunts) * 100
    print(f"    {type_doc:<30} : {nb:,} ({pourcentage:.0f}%)")
print()
print("  Langues des documents :")
for langue, nb in variete_langues.items():
    pourcentage = nb / len(emprunts) * 100
    print(f"    {langue:<30} : {nb:,} ({pourcentage:.0f}%)")
print()
print(f"  Nombre de filières différentes  : {nb_filieres}")
print(f"  Nombre de régions représentées  : {nb_regions}")
print()
print("  INTERPRÉTATION IDA :")
print(f"  → Le fonds présente {variete_types.nunique()} types documentaires différents.")
print(f"  → {variete_langues['Français']:,} documents en français, {variete_langues.get('Anglais',0):,} en anglais.")
print(f"  → Cette variété impose un schéma de métadonnées flexible (DCAT -- session S04).")
print(f"  → Défi IDA : harmoniser la description de types de documents très différents.")

# -- Graphique en camembert pour les types de documents --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variete_types.plot(kind='pie', ax=axes[0], autopct='%1.0f%%',
                   colors=['#3E1500','#BF360C','#E64A19','#FF7043','#FF8A65','#FFAB91'])
axes[0].set_title("Types de documents", fontweight='bold', color='#3E1500')
axes[0].set_ylabel("")

variete_langues.plot(kind='bar', ax=axes[1], color='#1565C0', edgecolor='white')
axes[1].set_title("Langues des documents", fontweight='bold', color='#3E1500')
axes[1].set_xlabel("Langue")
axes[1].set_ylabel("Nombre d'emprunts")
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle("V3 -- VARIÉTÉ : Diversité du fonds documentaire BU-UNG",
             fontsize=13, fontweight='bold', color='#3E1500', y=1.01)
plt.tight_layout()
plt.show()


---
## 💻 V4 -- VÉRACITÉ : Les données sont-elles fiables et complètes ?

### La notion en une phrase
La **Véracité** évalue la qualité, la fiabilité et la complétude des données -- c'est la dimension de l'audit qualité.

**Analogie IDA :** C'est l'équivalent du contrôle de catalogage : vérifier qu'une fiche n'est pas incomplète (zone obligatoire vide), incorrecte (date impossible) ou dupliquée (double saisie du même document).

### Pourquoi c'est critique pour l'IDA ?
Des données de mauvaise véracité publiées en open data nuisent à la réputation de l'institution et induisent des erreurs chez les réutilisateurs (chercheurs, ONG, journalistes).

### En pratique -- auditons la véracité de notre jeu 👇


In [ ]:
# ================================================================
# MESURE DE LA VÉRACITÉ -- V4 du Big Data
# ================================================================
# .isnull() = détecte les cellules vides (valeurs manquantes)
# .sum() = additionne le nombre de cellules vides par colonne
# .duplicated() = détecte les lignes entièrement dupliquées
# .describe() = calcule les statistiques de base d'une colonne numérique
# ================================================================

# -- 1. VALEURS MANQUANTES -- combien de cases vides ? --
nb_manquants_par_colonne = emprunts.isnull().sum()

print("V4 -- VÉRACITÉ : audit de qualité du jeu BU-UNG")
print("=" * 55)
print()
print("  1. VALEURS MANQUANTES (cases vides par colonne) :")
if nb_manquants_par_colonne.sum() == 0:
    print("     ✅ Aucune valeur manquante dans ce jeu de données.")
else:
    for col, nb in nb_manquants_par_colonne[nb_manquants_par_colonne > 0].items():
        print(f"     ⚠️  {col:<25} : {nb} cases vides")

print()

# -- 2. DOUBLONS -- y a-t-il des enregistrements identiques ? --
# .duplicated() renvoie True pour chaque ligne qui est un doublon
nb_doublons = emprunts.duplicated().sum()
print(f"  2. DOUBLONS (enregistrements identiques) : {nb_doublons}")
if nb_doublons == 0:
    print("     ✅ Aucun doublon détecté.")
else:
    print(f"     ⚠️  {nb_doublons} lignes dupliquées à supprimer.")

print()

# -- 3. VALEURS ABERRANTES -- des durées d'emprunt impossibles ? --
duree_min = emprunts["duree_jours"].min()
duree_max = emprunts["duree_jours"].max()
duree_moy = emprunts["duree_jours"].mean()
nb_suspects = (emprunts["duree_jours"] > 90).sum()   # Plus de 90 jours = suspect

print("  3. VALEURS ABERRANTES (durées d'emprunt en jours) :")
print(f"     Durée minimum      : {duree_min} jours")
print(f"     Durée maximum      : {duree_max} jours")
print(f"     Durée moyenne      : {duree_moy:.1f} jours")
print(f"     Durées > 90 jours  : {nb_suspects} (potentiellement aberrantes)")

print()
print("  INTERPRÉTATION IDA :")
print("  → La Véracité évalue la fiabilité AVANT toute publication.")
print("  → Un jeu bien vérifié inspire confiance aux réutilisateurs.")
print("  → Ces problèmes seront corrigés en session S05 (Qualité DAMA).")


---
## 💻 V5 -- VALEUR : Quelles décisions ce jeu de données permet-il ?

### La notion en une phrase
La **Valeur** est la connaissance actionnable qu'on peut extraire des données pour prendre des décisions concrètes.

**Analogie IDA :** Ce n'est pas simplement le fait d'avoir des statistiques -- c'est savoir **quoi acheter, quand ouvrir la bibliothèque, quelles filières prioriser** grâce à ces statistiques.

**C'est le V le plus important pour l'IDA** : sans valeur actionnable, collecter et stocker des données ne sert à rien.

### En pratique -- extrayons la valeur du jeu BU-UNG 👇


In [ ]:
# ================================================================
# EXTRACTION DE LA VALEUR -- V5 du Big Data
# ================================================================
# .groupby().size().sort_values() = compter et trier par groupe
# .idxmax() = trouver la valeur qui a le plus grand nombre
# ================================================================

# -- Question 1 : Quelle filière emprunte le plus ? --
# Utile pour : prioriser les acquisitions documentaires
emprunts_par_filiere = (emprunts
    .groupby("filiere")
    .size()
    .sort_values(ascending=False))

# -- Question 2 : Quel type de document est le plus emprunté ? --
# Utile pour : évaluer si le fonds répond aux besoins réels
top_type = emprunts["type_document"].value_counts().idxmax()
nb_top_type = emprunts["type_document"].value_counts().max()

# -- Question 3 : Quelles régions sont les plus représentées ? --
# Utile pour : adapter les horaires et les collections régionales
top_region = emprunts["region_origine"].value_counts().idxmax()

# -- Question 4 : Quel niveau d'étude emprunte le plus longtemps ? --
# Utile pour : ajuster la politique de prêt (durée autorisée)
duree_par_niveau = emprunts.groupby("niveau_etude")["duree_jours"].mean().sort_values(ascending=False)

print("V5 -- VALEUR : connaissances actionnables pour la direction BU-UNG")
print("=" * 65)
print()
print("  DÉCISION 1 -- Acquisitions prioritaires (filières les plus actives) :")
for filiere, nb in emprunts_par_filiere.head(3).items():
    print(f"    → {filiere:<30} : {nb:,} emprunts")
print(f"    Recommandation IDA : Renforcer le fonds documentaire en '{emprunts_par_filiere.index[0]}'.")
print()
print(f"  DÉCISION 2 -- Type de document le plus demandé :")
print(f"    → '{top_type}' avec {nb_top_type:,} emprunts")
print(f"    Recommandation IDA : Vérifier si le stock de '{top_type}' est suffisant.")
print()
print(f"  DÉCISION 3 -- Région d'origine dominante :")
print(f"    → '{top_region}' (usagers les plus nombreux)")
print(f"    Recommandation IDA : Adapter les horaires aux contraintes de transport de cette région.")
print()
print("  DÉCISION 4 -- Niveau d'étude et durée d'emprunt :")
for niveau, duree in duree_par_niveau.head(3).items():
    print(f"    → {niveau:<25} : {duree:.1f} jours en moyenne")


---
## 💻 V6 & V7 -- VARIABILITÉ et VISUALISATION

### V6 -- Variabilité : le contexte change le sens des données

Le même chiffre peut signifier des choses différentes selon le contexte. 50 emprunts en période d'examens = activité normale. 50 emprunts pendant les vacances = activité exceptionnelle.

**Analogie IDA :** Une notice de catalogage sans date de contexte perd une partie de sa valeur -- on ne sait plus si elle est récente ou ancienne.

### V7 -- Visualisation : rendre les données lisibles pour les décideurs

Un tableau de 5 000 lignes ne communique rien. Un graphique bien conçu permet à un directeur sans formation technique de prendre une décision en 30 secondes.

**Rôle IDA :** L'IDA conçoit les indicateurs et les tableaux de bord -- pas seulement l'informaticien.

### En pratique -- tableau de bord visuel 👇


In [ ]:
# ================================================================
# TABLEAU DE BORD BU-UNG -- V6 (Variabilité) et V7 (Visualisation)
# ================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -- Graphique 1 : Évolution des emprunts par mois (V6 -- saisonnalité) --
emprunts["mois"] = pd.to_datetime(emprunts["date_emprunt"]).dt.month
emprunts_par_mois = emprunts.groupby("mois").size()

noms_mois = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
emprunts_par_mois.index = noms_mois

emprunts_par_mois.plot(ax=axes[0,0], marker='o', color='#BF360C', linewidth=2)
axes[0,0].set_title("V6 -- Saisonnalité des emprunts (tous mois)", fontweight='bold', color='#3E1500')
axes[0,0].set_ylabel("Nombre d'emprunts")
axes[0,0].fill_between(range(12), emprunts_par_mois.values, alpha=0.2, color='#BF360C')

# -- Graphique 2 : Répartition par filière --
top5_filieres = emprunts["filiere"].value_counts().head(5)
top5_filieres.plot(kind='barh', ax=axes[0,1], color='#1565C0', edgecolor='white')
axes[0,1].set_title("V7 -- Top 5 filières les plus actives", fontweight='bold', color='#3E1500')
axes[0,1].set_xlabel("Nombre d'emprunts")

# -- Graphique 3 : Durée moyenne par niveau d'étude --
duree_niveau = emprunts.groupby("niveau_etude")["duree_jours"].mean().sort_values()
duree_niveau.plot(kind='bar', ax=axes[1,0], color='#558B2F', edgecolor='white')
axes[1,0].set_title("V7 -- Durée moyenne d'emprunt par niveau", fontweight='bold', color='#3E1500')
axes[1,0].set_ylabel("Jours (moyenne)")
axes[1,0].tick_params(axis='x', rotation=20)

# -- Graphique 4 : Répartition géographique des usagers --
emprunts["region_origine"].value_counts().plot(kind='bar', ax=axes[1,1],
                                                color='#E64A19', edgecolor='white')
axes[1,1].set_title("V7 -- Régions d'origine des usagers", fontweight='bold', color='#3E1500')
axes[1,1].set_ylabel("Nombre d'usagers")
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle("Tableau de bord BU-UNG -- Pour la direction de la bibliothèque",
             fontsize=14, fontweight='bold', color='#3E1500', y=1.01)
plt.tight_layout()
plt.show()

print("✅ Tableau de bord produit -- 4 graphiques, aucune ligne à modifier par l'IDA.")
print()
print("   RÔLE IDA ici : définir QUELS indicateurs sont pertinents pour la direction.")
print("   L'informaticien code -- l'IDA décide QUOI mesurer et COMMENT l'interpréter.")


---
## 🎯 Synthèse -- Le Rôle de l'IDA dans le Pipeline de Données

Les 7V décrivent le problème. L'IDA apporte la solution à 3 étapes clés :

| Étape du pipeline | Rôle technique | Rôle IDA |
|---|---|---|
| **Collecte** | Enregistrement dans le SIGB | Définir quels champs collecter, les règles qualité, les droits |
| **Traitement** | Python, OpenRefine | Définir ce qui est "valide", corriger les erreurs métier |
| **Publication** | CKAN, portail web | Choisir la licence, rédiger les métadonnées DCAT, fixer les accès |

L'IDA n'est pas l'exécutant technique -- il est le **garant de la pertinence et de la qualité documentaire** à chaque étape.

---


---
## ✏️ EXERCICE -- Appliquez les 7V à votre propre institution

**Consigne :** Complétez le tableau ci-dessous en remplaçant les zones `[...]`.
Choisissez un jeu de données réel ou réaliste de votre institution.


In [ ]:
# ================================================================
# EXERCICE -- Analyse 7V de votre institution
# ================================================================
# Modifiez UNIQUEMENT les valeurs entre guillemets ci-dessous.
# Remplacez [À COMPLÉTER] par vos propres réponses.
# ================================================================

# -- Identifiez votre institution et votre jeu de données --
NOM_INSTITUTION  = "[À COMPLÉTER : ex. Archives Nationales du Cameroun]"
JEU_DE_DONNEES   = "[À COMPLÉTER : ex. Registre des versements 2010-2023]"

# -- Estimez chaque V de 0 à 10 (0 = absent/faible, 10 = très fort) --
mes_7V = {
    "V1 -- Volume"       : None,   # Remplacez None par une valeur entre 0 et 10
    "V2 -- Vélocité"     : None,
    "V3 -- Variété"      : None,
    "V4 -- Véracité"     : None,
    "V5 -- Valeur"       : None,
    "V6 -- Variabilité"  : None,
    "V7 -- Visualisation": None,
}

# -- Pour chaque V, expliquez votre note en une phrase --
mes_justifications = {
    "V1 -- Volume"       : "[À COMPLÉTER : ex. 45 000 versements, trop grand pour Excel]",
    "V2 -- Vélocité"     : "[À COMPLÉTER : ex. 200 nouvelles entrées par mois]",
    "V3 -- Variété"      : "[À COMPLÉTER : ex. papier + numérique + audiovisuel]",
    "V4 -- Véracité"     : "[À COMPLÉTER : ex. 30% de fiches incomplètes]",
    "V5 -- Valeur"       : "[À COMPLÉTER : ex. décisions d'élimination, valorisation]",
    "V6 -- Variabilité"  : "[À COMPLÉTER : ex. signification différente selon la période]",
    "V7 -- Visualisation": "[À COMPLÉTER : ex. aucun tableau de bord actuellement]",
}

# -- Affichage automatique de votre analyse --
print(f"ANALYSE 7V -- {NOM_INSTITUTION}")
print(f"Jeu de données : {JEU_DE_DONNEES}")
print("=" * 65)
print()

notes_valides = {k:v for k,v in mes_7V.items() if isinstance(v,(int,float)) and 0<=v<=10}

for v_nom in mes_7V:
    note = mes_7V[v_nom]
    justif = mes_justifications[v_nom]
    if note is not None and not "[À COMPLÉTER]" in justif:
        barre = "█" * int(note) + "░" * (10 - int(note))
        print(f"  {v_nom:<25} : {note}/10  {barre}")
        print(f"    → {justif}")
        print()
    else:
        print(f"  {v_nom:<25} : [À COMPLÉTER]")

if notes_valides:
    score_moyen = sum(notes_valides.values()) / len(notes_valides)
    print(f"Score Big Data moyen : {score_moyen:.1f} / 10")

# Sauvegarde automatique si l'exercice est complété
if notes_valides and NOM_INSTITUTION != "[À COMPLÉTER : ex. Archives Nationales du Cameroun]":
    rapport = {
        "institution": NOM_INSTITUTION,
        "jeu_de_donnees": JEU_DE_DONNEES,
        "notes_7V": mes_7V,
        "justifications": mes_justifications,
        "date_analyse": datetime.now().strftime("%Y-%m-%d")
    }
    with open("MIDA507_S01_rapport_7V.json", "w", encoding="utf-8") as fichier:
        json.dump(rapport, fichier, ensure_ascii=False, indent=2)
    print()
    print("✅ Rapport sauvegardé : MIDA507_S01_rapport_7V.json")
    print("   Partagez ce fichier avec votre enseignant comme livrable de session.")


---
## ✅ Bilan de la Session 01

### Ce que vous avez accompli

| Compétence | Acquise |
|---|---|
| Définir le Big Data et expliquer pourquoi il concerne l'IDA | ✅ |
| Mesurer le Volume (V1) d'un jeu de données | ✅ |
| Analyser la Vélocité (V2) : emprunts par année | ✅ |
| Décrire la Variété (V3) : types, langues, filières | ✅ |
| Auditer la Véracité (V4) : valeurs manquantes, doublons, aberrants | ✅ |
| Extraire la Valeur (V5) : décisions actionnables | ✅ |
| Produire un tableau de bord (V7) pour la direction | ✅ |
| Appliquer les 7V à mon institution (exercice) | 🟡 À compléter |

### Ce qui vous attend en Session 02
Nous allons apprendre à **gérer le cycle de vie complet** de ce jeu de données :
qui l'a créé, comment il a évolué, comment garantir son intégrité, et comment évaluer s'il est conforme aux standards FAIR (Findable, Accessible, Interoperable, Reusable).

---

*Notebook MIDA507 S01 -- Master MIDA -- Université de Douala*
